In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path().resolve()))

import torch
import config
from models.embedding import *
from models.student_model import *
from models.teacher_model import *
from training.baseline_train import *
from training.distill_train import *
from distill_loss import *
from data.create_dataloader import *
train_load, val_load, test_load = load_and_create_dataloaders()

In [ ]:
print(config.DEVICE)

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path().resolve()))

import torch
import config
from models.embedding import *
from models.student_model import *
from models.teacher_model import *
from training.baseline_train import *
from training.distill_train import *
from distill_loss import *
from data.create_dataloader import *
if __name__ == '__main__':
    train_load, val_load, test_load = load_and_create_dataloaders()
    
    hist = train_distillation(
        teacher=teacher_model(),
        student=create_student('custom'),
        train_loader=train_load,
        val_loader=val_load,
        test_loader = test_load,
        teacher_name='cifar10_resnet56',
        student_name='custom',
        epochs=5,
        alpha = 0.6
    )

In [ ]:

CHECKPOINT_DIR = Path("saved_models")
CHECKPOINT_DIR.mkdir(exist_ok=True)

print('test')

train_loader, val_loader, test_loader = load_and_create_dataloaders()

teacher = teacher_model()

# 3. Student factories
students = {
    "custom": lambda: create_student('custom')
}

shared_kwargs = {
    "epochs": config.NUM_EPOCHS,
    "lr": config.LEARNING_RATE,
    "train_loader": train_loader,
    "val_loader": val_loader,
    "test_loader": test_loader
}

print('test2')

trained_models = {}

for name, factory in students.items():
    print(f"\n{'='*40}")
    print(f"STUDENT: {name.upper()}")
    
    if(name != 'resnet32'):
        print("Distill")
        print(config.DEVICE)
        distill_model = factory().to(config.DEVICE)
        train_distillation(
            teacher=teacher, student=distill_model, student_name=name,
            alpha=0.6, distill_type="cosine", **shared_kwargs
        )
        # Distillation
        trained_models[f"{name}_distill"] = distill_model
        torch.save(distill_model.state_dict(), CHECKPOINT_DIR / f"{name}_distill.pth")
    # Baseline
    print("Baseline")
    baseline_model = factory().to(config.DEVICE)
    train_baseline(model=baseline_model, model_name=name, **shared_kwargs)
    trained_models[f"{name}_baseline"] = baseline_model
    torch.save(baseline_model.state_dict(), CHECKPOINT_DIR / f"{name}_baseline.pth")
    

print("done")

In [ ]:
import sys
from pathlib import Path
import torch
import wandb
import matplotlib.pyplot as plt

# Zapewnienie widoczności modułów
sys.path.insert(0, str(Path().resolve()))

import config
from models.teacher_model import teacher_model
from models.student_model import create_student
from data.create_dataloader import load_and_create_dataloaders
from training.wandb_log import log_pair_embeddings

# 1. Inicjalizacja środowiska i Dataloadera
# Używamy reinit=True, aby móc odpalać komórkę wielokrotnie
wandb.init(project="distill_uni_proj", name="notebook_visualizations", mode="online", reinit=True)
CHECKPOINT_DIR = Path("saved_models")

print("Ładowanie danych walidacyjnych...")
_, val_loader, _ = load_and_create_dataloaders()

# 2. Tworzenie instancji modeli (puste architektury)
print("Przygotowywanie architektur modeli...")
teacher = teacher_model().to(config.DEVICE)

student_name = "custom"  # Śmiało zmień na "cnn_small" lub "resnet32"
baseline_student = create_student(student_name).to(config.DEVICE)
distill_student = create_student(student_name).to(config.DEVICE)

# 3. Wypełnianie modeli Twoimi wyuczonymi wagami
print("Ładowanie wyuczonych wag z folderu saved_models...")
baseline_path = CHECKPOINT_DIR / f"{student_name}_baseline.pth"
distill_path = CHECKPOINT_DIR / f"{student_name}_distill.pth"

if baseline_path.exists() and distill_path.exists():
    baseline_student.load_state_dict(torch.load(baseline_path, map_location=config.DEVICE))
    distill_student.load_state_dict(torch.load(distill_path, map_location=config.DEVICE))
else:
    print(f"⚠️ UWAGA: Brakuje plików .pth dla {student_name}!")

# 4. Generowanie i wyświetlanie wykresów
print("\n--- Generowanie wykresów PCA i t-SNE ---")

# A) Baseline vs Distillation
print("1. Porównanie standardowego uczenia z destylacją")
log_pair_embeddings(baseline_student, distill_student, val_loader, 
                    "Baseline", "Distillation", f"{student_name}_base_vs_dist", method="PCA")
log_pair_embeddings(baseline_student, distill_student, val_loader, 
                    "Baseline", "Distillation", f"{student_name}_base_vs_dist", method="t-SNE")

# B) Teacher vs Student (Baseline)
print("2. Porównanie Nauczyciela z Uczniem (Baseline)")
log_pair_embeddings(teacher, baseline_student, val_loader, 
                    "Teacher", "Student_Baseline", f"teacher_vs_{student_name}_base", method="PCA")
log_pair_embeddings(teacher, baseline_student, val_loader, 
                    "Teacher", "Student_Baseline", f"teacher_vs_{student_name}_base", method="t-SNE")

# C) Teacher vs Student (Distillation)
print("3. Porównanie Nauczyciela z Uczniem (Distillation)")
log_pair_embeddings(teacher, distill_student, val_loader, 
                    "Teacher", "Student_Distill", f"teacher_vs_{student_name}_dist", method="PCA")
log_pair_embeddings(teacher, distill_student, val_loader, 
                    "Teacher", "Student_Distill", f"teacher_vs_{student_name}_dist", method="t-SNE")

wandb.finish()
print("Koniec pracy. Wykresy powinny być widoczne powyżej!")

In [ ]:
hist = train_distillation(teacher_model(), 
create_student('resnet20'), 
train_load, 
val_load, 
epochs = 30, 
teacher_name = 'cifar10_resnet56', 
student_name = 'resnet20'
)
print(hist)